# 학습 곡선 실습

**Learning Curve**

데이터 양이나 학습 진행에 따라 성능이 변하는 추이를 그린 곡선.

소재 분야에서 이해하기: 데이터를 더 모으는 것이 성능에 도움이 될지 판단한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [scikit-learn 교차검증 문서](https://scikit-learn.org/stable/modules/cross_validation.html)

## 1. 데이터를 더 모으는 게 답일까

학습 곡선은 그 판단의 근거가 됩니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

def make_alloy_data(n=240, noise=6.0, seed=0):
    """개념 확인용 합성 데이터. 실제 합금 측정값이 아닙니다.

    x1 소성 온도(600-900 C), x2 유지 시간(0.5-8 h), x3 첨가 원소 비율(0-5 at%),
    x4 측정 노이즈만 담긴 무의미한 변수. y 는 경도(HV) 를 흉내낸 값입니다.
    """
    rng = np.random.default_rng(seed)
    x1 = rng.uniform(600, 900, n)
    x2 = rng.uniform(0.5, 8.0, n)
    x3 = rng.uniform(0.0, 5.0, n)
    x4 = rng.normal(0.0, 1.0, n)
    y = (120 + 0.14 * (x1 - 600) + 9.0 * np.sqrt(x2) + 11.0 * x3
         - 0.9 * x3 ** 2 - 0.004 * (x1 - 750) * x2 + rng.normal(0, noise, n))
    X = np.column_stack([x1, x2, x3, x4])
    return X, y, ['소성온도', '유지시간', '첨가비율', '무관변수']


X, y, FEATURES = make_alloy_data()
print(X.shape, y.shape, FEATURES)
print('경도 평균 %.1f, 표준편차 %.1f' % (y.mean(), y.std()))

In [ ]:
from sklearn.model_selection import learning_curve, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

for name, model in [('random forest', RandomForestRegressor(n_estimators=200, random_state=0)),
                    ('linear', LinearRegression())]:
    sizes, train_scores, valid_scores = learning_curve(
        model, X, y, train_sizes=np.linspace(0.1, 1.0, 8),
        cv=KFold(5, shuffle=True, random_state=0), scoring='neg_mean_absolute_error')
    plt.plot(sizes, -train_scores.mean(1), 'o--', label=name + ' (train)')
    plt.plot(sizes, -valid_scores.mean(1), 's-', label=name + ' (valid)')
    print('%-14s 데이터 %d개에서 검증 MAE %.2f -> %d개에서 %.2f'
          % (name, sizes[0], -valid_scores.mean(1)[0], sizes[-1], -valid_scores.mean(1)[-1]))
plt.xlabel('training samples'); plt.ylabel('MAE (HV)'); plt.legend(); plt.show()

## 2. 해석

검증 곡선이 아직 내려가는 중이면 데이터를 더 모으는 것이 효과적입니다. 이미 평평해졌다면
데이터보다 기술자나 모델을 바꾸는 편이 낫습니다. 학습 곡선과 검증 곡선의 간격은 과적합의 정도를 보여줍니다.

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#learning-curve)을 여세요.